# Importing Relevant Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
import sklearn
import nltk
from tqdm.auto import tqdm

# Normalization and Initial Cleaning

In [ ]:
import pandas as pd
import os

data_directory = '/content/drive/My Drive/'
output_combined_csv_path = os.path.join(data_directory, 'combined_shopee_reviews_raw.csv')

# Load the combined CSV file into a DataFrame
df = pd.read_csv(output_combined_csv_path)

print(f"Combined DataFrame loaded successfully from: {output_combined_csv_path}")
print(f"Shape of loaded DataFrame: {df.shape}")
display(df.head())

Combined DataFrame loaded successfully from: /content/drive/My Drive/combined_shopee_reviews_raw.csv
Shape of loaded DataFrame: (2994050, 4)


,review,rating,date,thumbs_up
0,bisa ga sih iklannya di benerin. jangan maksa ...,1,2025-12-30 23:56:40,5
1,amanah,5,2025-12-30 23:56:02,0
2,"Makin hari pengirimannya makin lama, jadi male...",1,2025-12-30 23:55:19,0
3,"sangat bantu belanja ku si ini, tapi ngeselin ...",3,2025-12-30 23:54:48,0
4,mantap,5,2025-12-30 23:51:59,0


In [ ]:
# Displays all column names in a DataFrame
print(df.columns)

Index(['review', 'rating', 'date', 'thumbs_up'], dtype='object')


 ## Including:
    Initial normalization before Aspect Detection and Manual Annotation.

    Steps:
    1. Convert to string
    2. Lowercase
    3. Remove URLs
    4. Remove @mentions
    5. Remove hashtags (# only, keep the word)
    6. Remove emojis / non-ASCII characters
    7. Remove control characters
    8. Normalize whitespace


In [ ]:
import re
import pandas as pd
import unicodedata

# Cleaning Function
def initial_clean(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Unicode Normalization
    text = unicodedata.normalize("NFKC", text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 3. Remove username mentions
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only
    text = re.sub(r'#(\w+)', r'\1', text)

    # 5. Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 6. Remove control characters
    text = re.sub(r'[\r\n\t\f\v]', ' ', text)

    # 7. Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply Cleaning
df['review_clean'] = df['review'].apply(initial_clean)


# Remove Duplicate Reviews
before = len(df)

df = df.drop_duplicates(subset=['review_clean']).reset_index(drop=True)

after = len(df)

print("=" * 50)
print("INITIAL CLEANING COMPLETED")
print("=" * 50)
print(f"Original data      : {before}")
print(f"After duplicates   : {after}")
print(f"Duplicates removed : {before-after}")
print("=" * 50)

INITIAL CLEANING COMPLETED
Original data      : 2994050
After duplicates   : 1743710
Duplicates removed : 1250340


In [ ]:
# Preview
df[['review', 'review_clean']].head(10)

,review,review_clean
0,bisa ga sih iklannya di benerin. jangan maksa ...,bisa ga sih iklannya di benerin. jangan maksa ...
1,amanah,amanah
2,"Makin hari pengirimannya makin lama, jadi male...","makin hari pengirimannya makin lama, jadi male..."
3,"sangat bantu belanja ku si ini, tapi ngeselin ...","sangat bantu belanja ku si ini, tapi ngeselin ..."
4,mantap,mantap
5,saya mendapatkan produk dengan harga yg bersah...,saya mendapatkan produk dengan harga yg bersah...
6,belanja di shopee dari dulu sangat baik dan me...,belanja di shopee dari dulu sangat baik dan me...
7,lumayan lah tapi gara² ada iklan???,lumayan lah tapi gara2 ada iklan???
8,"kebocoran data, penipu punya data pribadi kita...","kebocoran data, penipu punya data pribadi kita..."
9,Paylater saya di 0 kan,paylater saya di 0 kan


In [ ]:
# Save Initial Clean Dataset

from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/ABSA_Shopee/02_initial_clean.parquet"

import os
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

df.to_parquet(SAVE_PATH, index=False)

print("=" * 50)
print("Initial cleaned dataset saved successfully!")
print(f"Location : {SAVE_PATH}")
print("=" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initial cleaned dataset saved successfully!
Location : /content/drive/MyDrive/ABSA_Shopee/02_initial_clean.parquet


# Load if Runtime Off

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

LOAD_PATH = "/content/drive/MyDrive/ABSA_Shopee/02_initial_clean.parquet"

df = pd.read_parquet(LOAD_PATH)

print(df.shape)

df.head()

Mounted at /content/drive
(1743710, 5)


,review,rating,date,thumbs_up,review_clean
0,bisa ga sih iklannya di benerin. jangan maksa ...,1,2025-12-30 23:56:40,5,bisa ga sih iklannya di benerin. jangan maksa ...
1,amanah,5,2025-12-30 23:56:02,0,amanah
2,"Makin hari pengirimannya makin lama, jadi male...",1,2025-12-30 23:55:19,0,"makin hari pengirimannya makin lama, jadi male..."
3,"sangat bantu belanja ku si ini, tapi ngeselin ...",3,2025-12-30 23:54:48,0,"sangat bantu belanja ku si ini, tapi ngeselin ..."
4,mantap,5,2025-12-30 23:51:59,0,mantap


# Anchor Filtering

In [ ]:
import re
import pandas as pd

# STRONG ANCHOR
# Almost always showing gamification


STRONG_ANCHORS = [

    # ---------- Points ----------
    "koin",
    "coin",
    "coins",
    "point",
    "points",
    "poin",
    "shopee coin",
    "shopee coins",
    "shopee koin",

    # ---------- Mission ----------
    "misi",
    "mission",
    "missions",
    "quest",
    "quests",
    "task",
    "daily mission",
    "daily quest",
    "weekly mission",

    # ---------- Chance ----------
    "spin",
    "lucky spin",
    "spin wheel",
    "wheel",
    "gacha",
    "draw",

    # ---------- Progress ----------
    "level",
    "level up",
    "tier",
    "rank",
    "ranking",
    "leaderboard",
    "leaderboards",

    # ---------- Check In ----------
    "check in",
    "check-in",
    "checkin",
    "cek in",
    "cekin",
    "daily check in",
    "daily check-in",
    "daily checkin",
    "login reward",
    "daily reward",
    "daily login",
    "absen",


    # ---------- Membership ----------
    "vip",
    "vip member",
    "membership tier",
    "member gold",
    "member platinum",
    "member silver",


    # ---------- Shopee Games ----------
    "shopee games",
    "shopee game",
    "shopee tanam",
    "tanam",
    "shopee capit",
    "capit",
    "shopee candy",
    "candy",
    "tebak kata",
    "shopee blockzi"
    "blockzi",
    "cocoki",
    "ceki ceki",
    "nego neko",
    "fruity",
    "block",
    "game blok",
    "siram",

]


# WEAK ANCHOR
# Need context
WEAK_ANCHORS = [

    "game",
    "main",
    "bermain",


    "reward",
    "hadiah",
    "bonus",

    "voucher",
    "kupon",

    "claim",
    "klaim",
    "redeem",

    "event",
    "acara",

    "progress",
    "progres",

    "naik",

    "teman",
    "invite",
    "ajak",

    "team",
    "tim",

    "collect",
    "collection",
    "koleksi",

    "unlock",
    "terbuka",

    "achievement",
    "badge",

    "gift"

]


# EXCLUSION
EXCLUSION = [

    # Paylater
    "paylater",
    "spaylater",
    "pinjaman",
    "spinjam",
    "cicilan",
    "hutang",

    # Logistics
    "kurir",
    "pengiriman",
    "paket",
    "resi",
    "tracking",
    "ekspedisi",
    "jne",
    "jnt",
    "sicepat",
    "anteraja",
    "spx",

    # Shopping
    "checkout",
    "seller",
    "penjual",
    "barang",
    "produk",
    "pesanan",
    "order",
    "refund",


    # Payment
    "shopeepay",
    "spay",
    "transfer",
    "rekening",
    "bank",
    "cod",
    "cash on delivery",
    "ongkir",
    "promo toko",
    "flash sale",
    "diskon",
    "voucher toko"


]

# Compile regex
def compile_patterns(words):

    patterns = []

    for w in words:

        patterns.append(
            re.compile(
                r"\b" + re.escape(w) + r"\b"
            )
        )

    return patterns


strong_patterns = compile_patterns(STRONG_ANCHORS)

weak_patterns = compile_patterns(WEAK_ANCHORS)

exclude_patterns = compile_patterns(EXCLUSION)


# Filtering Function

def anchor_filter(text):

    text = str(text)

    strong_found = []

    weak_found = []

    exclude_found = []

    for word, pattern in zip(STRONG_ANCHORS, strong_patterns):

        if pattern.search(text):
          strong_found.append(word)

    for word, pattern in zip(WEAK_ANCHORS, weak_patterns):
        if pattern.search(text):
          weak_found.append(word)

    for word, pattern in zip(EXCLUSION, exclude_patterns):
        if pattern.search(text):
          exclude_found.append(word)

    # Rule 1
    if len(strong_found) > 0:

        return pd.Series([
            True,
            strong_found,
            weak_found,
            exclude_found
        ])

    # Rule 2
    if len(weak_found) >= 2:

        return pd.Series([
            True,
            strong_found,
            weak_found,
            exclude_found
        ])

    # Rule 3
    return pd.Series([
        False,
        strong_found,
        weak_found,
        exclude_found
    ])


# Run Filtering

df[
    [
        "is_gamification",
        "strong_anchor",
        "weak_anchor",
        "exclude_keyword"
    ]
] = df["review_clean"].apply(anchor_filter)


# Keep Candidate Reviews

candidate_df = df[
    df["is_gamification"]
].copy()

candidate_df.reset_index(
    drop=True,
    inplace=True
)

print("="*60)
print("ANCHOR FILTERING FINISHED")
print("="*60)
print("Original Reviews :", len(df))
print("Candidate Reviews:", len(candidate_df))
print("="*60)

candidate_df.head()

ANCHOR FILTERING FINISHED
Original Reviews : 1743710
Candidate Reviews: 26552


,review,rating,date,thumbs_up,review_clean,is_gamification,strong_anchor,weak_anchor,exclude_keyword
0,saya mendapatkan 380 koin ya bos,5,2025-12-30 23:18:54,0,saya mendapatkan 380 koin ya bos,True,[koin],[],[]
1,"Shopee iklan nya dimana mana, lagi main game p...",2,2025-12-30 22:01:46,0,"shopee iklan nya dimana mana, lagi main game p...",True,[],"[game, main]",[]
2,Alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,0,alhamdulillah dapat koin 50 semoga lebih besar...,True,[koin],[],[]
3,cara mudah belanja dan mudah untuk mendapatkan...,5,2025-12-30 16:56:57,0,cara mudah belanja dan mudah untuk mendapatkan...,True,[koin],[],[]
4,"shopee skr payah ga kayak dulu, sekarang vouch...",1,2025-12-30 15:03:49,0,"shopee skr payah ga kayak dulu, sekarang vouch...",True,[vip],[voucher],"[spaylater, diskon]"


# Import Gamification Dictionary

In [ ]:
!find /content/drive -name "gamification_dictionary.py"

/content/drive/MyDrive/gamification_dictionary.py


In [ ]:
import sys

sys.path.append("/content/drive/MyDrive")

In [ ]:
from gamification_dictionary import gamification_dictionary

# Mapping into 30 Elements Gamification Based on Gamification Dictionary

In [ ]:
print(type(gamification_dictionary))
print(len(gamification_dictionary))
print(gamification_dictionary.keys())

<class 'dict'>
30
dict_keys(['constraints', 'emotions', 'narrative', 'progression', 'relationships', 'challenges', 'chance', 'competition', 'cooperation', 'feedback', 'resource_acquisition', 'rewards', 'in_game_transactions', 'turns', 'win_states', 'achievements', 'avatar', 'badges', 'boss_fights', 'collections', 'combat', 'content_unlocking', 'gifting', 'leaderboards', 'levels', 'points', 'quests', 'social_graph', 'teams', 'virtual_goods'])


In [ ]:
import re
import pandas as pd
from gamification_dictionary import gamification_dictionary

### Compile Regex

In [ ]:
compiled_dictionary = {}

for aspect, info in gamification_dictionary.items():

    patterns = []

    for keyword in info["keywords"]:

        pattern = re.compile(
            r"\b" + re.escape(keyword) + r"\b",
            flags=re.IGNORECASE
        )

        patterns.append((keyword, pattern))

    compiled_dictionary[aspect] = {

        "category": info["category"],

        "description": info["description"],

        "patterns": patterns

    }

print("Dictionary compiled successfully!")

Dictionary compiled successfully!


# Detecting Aspects

In [ ]:
def detect_aspects(text):

    text = str(text)

    detected = []

    for aspect, info in compiled_dictionary.items():

        matched = []

        score = 0

        for keyword, pattern in info["patterns"]:

            if pattern.search(text):

                matched.append(keyword)

                # phrase lebih panjang = skor lebih tinggi
                if len(keyword.split()) >= 2:
                    score += 3
                else:
                    score += 2

        if score > 0:

            detected.append({

                "aspect": aspect,

                "category": info["category"],

                "matched_keywords": matched,

                "score": score

            })

    return detected

In [ ]:
review = "Saya dapat koin dari daily mission tapi hadiah lucky spin jelek"

detect_aspects(review)

[{'aspect': 'chance',
  'category': 'Mechanics',
  'matched_keywords': ['lucky spin', 'spin'],
  'score': 5},
 {'aspect': 'resource_acquisition',
  'category': 'Mechanics',
  'matched_keywords': ['dapat koin'],
  'score': 3},
 {'aspect': 'rewards',
  'category': 'Mechanics',
  'matched_keywords': ['hadiah'],
  'score': 2},
 {'aspect': 'points',
  'category': 'Components',
  'matched_keywords': ['koin'],
  'score': 2},
 {'aspect': 'quests',
  'category': 'Components',
  'matched_keywords': ['daily mission', 'mission'],
  'score': 5}]

## Review Aspect Pair Generator

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

pair_rows = []

for idx, row in tqdm(candidate_df.iterrows(), total=len(candidate_df)):

    review = row["review"]
    review_clean = row["review_clean"]

    detected_aspects = detect_aspects(review_clean)

    if len(detected_aspects) == 0:
        continue

    for item in detected_aspects:

        pair_rows.append({

            # ---------- Original Information ----------
            "review_id": idx,
            "review": review,
            "review_clean": review_clean,
            "rating": row["rating"],
            "date": row["date"],
            "thumbs_up": row["thumbs_up"],

            # ---------- Aspect Detection ----------
            "aspect": item["aspect"],
            "aspect_category": item["category"],
            "matched_keywords": ", ".join(item["matched_keywords"]),
            "confidence_score": item["score"],

            # ---------- Manual Annotation ----------
            "is_gamification_final": "Yes",
            "aspect_verified": item["aspect"],
            "aspect_changed": "No",
            "sentiment": "",
            "notes": "",

        })

pair_df = pd.DataFrame(pair_rows)

print("="*60)
print("REVIEW-ASPECT PAIR FINISHED")
print("="*60)
print("Original Candidate Reviews :", len(candidate_df))
print("Generated Aspect Pairs    :", len(pair_df))
print("="*60)

pair_df.head()

  0%|          | 0/26552 [00:00<?, ?it/s]

REVIEW-ASPECT PAIR FINISHED
Original Candidate Reviews : 26552
Generated Aspect Pairs    : 24714


,review_id,review,review_clean,rating,date,thumbs_up,aspect,aspect_category,matched_keywords,confidence_score,is_gamification_final,aspect_verified,aspect_changed,sentiment,notes
0,0,saya mendapatkan 380 koin ya bos,saya mendapatkan 380 koin ya bos,5,2025-12-30 23:18:54,0,points,Components,koin,2,Yes,points,No,,
1,2,Alhamdulillah dapat koin 50 semoga lebih besar...,alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,0,resource_acquisition,Mechanics,dapat koin,3,Yes,resource_acquisition,No,,
2,2,Alhamdulillah dapat koin 50 semoga lebih besar...,alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,0,points,Components,koin,2,Yes,points,No,,
3,3,cara mudah belanja dan mudah untuk mendapatkan...,cara mudah belanja dan mudah untuk mendapatkan...,5,2025-12-30 16:56:57,0,points,Components,koin,2,Yes,points,No,,
4,5,"saya masih berlangganan shoopie VIP,tp entah k...","saya masih berlangganan shoopie vip,tp entah k...",4,2025-12-30 13:52:04,0,emotions,Dynamics,kesal,2,Yes,emotions,No,,


In [ ]:
pair_df = pair_df.sort_values(
    by=[
        "review_id",
        "aspect_category",
        "confidence_score"
    ],
    ascending=[
        True,
        True,
        False
    ]
).reset_index(drop=True)

pair_df.head(20)

,review_id,review,review_clean,rating,date,thumbs_up,aspect,aspect_category,matched_keywords,confidence_score,is_gamification_final,aspect_verified,aspect_changed,sentiment,notes
0,0,saya mendapatkan 380 koin ya bos,saya mendapatkan 380 koin ya bos,5,2025-12-30 23:18:54,0,points,Components,koin,2,Yes,points,No,,
1,2,Alhamdulillah dapat koin 50 semoga lebih besar...,alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,0,points,Components,koin,2,Yes,points,No,,
2,2,Alhamdulillah dapat koin 50 semoga lebih besar...,alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,0,resource_acquisition,Mechanics,dapat koin,3,Yes,resource_acquisition,No,,
3,3,cara mudah belanja dan mudah untuk mendapatkan...,cara mudah belanja dan mudah untuk mendapatkan...,5,2025-12-30 16:56:57,0,points,Components,koin,2,Yes,points,No,,
4,5,"saya masih berlangganan shoopie VIP,tp entah k...","saya masih berlangganan shoopie vip,tp entah k...",4,2025-12-30 13:52:04,0,emotions,Dynamics,kesal,2,Yes,emotions,No,,
5,6,Apa sekarang menonton live di shopee sudah tid...,apa sekarang menonton live di shopee sudah tid...,5,2025-12-30 10:59:29,1234,points,Components,koin,2,Yes,points,No,,
6,6,Apa sekarang menonton live di shopee sudah tid...,apa sekarang menonton live di shopee sudah tid...,5,2025-12-30 10:59:29,1234,rewards,Mechanics,hadiah,2,Yes,rewards,No,,
7,7,Sangat-sangat menyenangkan belanja di shopee.....,sangat-sangat menyenangkan belanja di shopee.....,5,2025-12-30 10:11:55,10,narrative,Dynamics,shopee tanam,3,Yes,narrative,No,,
8,9,entah kenapa akun saya sekarang ga bisa klaim ...,entah kenapa akun saya sekarang ga bisa klaim ...,4,2025-12-30 04:28:22,0,points,Components,koin,2,Yes,points,No,,
9,10,"ini koin bisa gak di tarik kedana,,","ini koin bisa gak di tarik kedana,,",5,2025-12-30 03:04:22,0,points,Components,koin,2,Yes,points,No,,


# Count Pair Aspects

In [ ]:
print(pair_df["aspect"].value_counts())

aspect
points                  12085
rewards                  2597
emotions                 2443
narrative                2335
resource_acquisition     1299
challenges                987
win_states                648
teams                     316
quests                    309
turns                     266
in_game_transactions      243
relationships             225
collections               169
chance                    160
content_unlocking         133
progression               127
constraints                97
social_graph               75
competition                47
cooperation                37
levels                     35
gifting                    32
feedback                   18
boss_fights                15
achievements                4
virtual_goods               4
combat                      3
leaderboards                2
avatar                      2
badges                      1
Name: count, dtype: int64


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/ABSA_Review_Aspect_Pairs_Raw.xlsx"

pair_df.to_excel(
    SAVE_PATH,
    index=False
)

print("Saved to:")
print(SAVE_PATH)

Saved to:
/content/drive/MyDrive/ABSA_Review_Aspect_Pairs_Raw.xlsx


# Count Matched Keywords

In [ ]:
pair_df[
    pair_df["aspect"]=="points"
]["matched_keywords"].value_counts().head(30)

,count
matched_keywords,
koin,9408
poin,1053
coin,721
point,542
"koin, poin",86
"koin, coin",69
"shopee koin, koin",50
"koin, koin hilang",43
"shopee coin, coin",21


In [ ]:
pair_df[
    pair_df["aspect"]=="narrative"
]["matched_keywords"].value_counts().head(30)

,count
matched_keywords,
shopee tanam,837
shopee game,312
shopee games,281
cocoki,260
shopee capit,224
shopee candy,174
tebak kata,63
ceki ceki,38
nego neko,34


In [ ]:
pair_df[
    pair_df["aspect"]=="teams"
]["matched_keywords"].value_counts().head(30)

,count
matched_keywords,
tim,190
team,97
grup,13
"team, tim",11
group,5


In [ ]:
pair_df[
    pair_df["aspect"]=="levels"
]["matched_keywords"].value_counts()

,count
matched_keywords,
level member,25
gold member,3
upgrade member,3
platinum member,2
silver member,1
vip member,1


In [ ]:
pair_df[
    pair_df["aspect"]=="chance"
]["matched_keywords"].value_counts()

,count
matched_keywords,
spin,50
gacha,41
putar,15
draw,11
putaran,9
acak,8
hoki,8
luck,6
"spin, putar",3


In [ ]:
pair_df.shape

(24714, 15)

# Candidate Aspect Mapping

In [ ]:
# Dictionary mapping anchor -> Werbach-Hunter Aspect
# Used as SUGGESTION, not final label

aspect_mapping = {

    # COMPONENTS
    "Points":[
        "koin","coin","coins","point","points","poin",
        "shopee coin","shopee coins","shopee koin"
    ],

    "Levels":[
        "level","tier","vip",
        "vip member",
        "member gold",
        "member silver",
        "member platinum",
        "membership tier"
    ],

    "Leaderboards":[
        "leaderboard",
        "leaderboards",
        "ranking",
        "rank"
    ],

    "Achievements":[
        "achievement",
        "badge"
    ],

    "Quests":[
        "quest",
        "quests"
    ],


    # MECHANICS
    "Challenges":[
        "misi",
        "mission",
        "missions",
        "task",
        "daily mission",
        "daily quest",
        "weekly mission"
    ],

    "Chance":[
        "spin",
        "lucky spin",
        "wheel",
        "spin wheel",
        "gacha",
        "draw"
    ],

    "Rewards":[
        "reward",
        "hadiah",
        "bonus",
        "voucher",
        "kupon",
        "claim",
        "klaim",
        "redeem",
        "daily reward",
        "login reward"
    ],

    "Feedback":[
        "progress",
        "progres"
    ],

    "Relationships":[
        "invite",
        "ajak",
        "teman"
    ],


    # DYNAMICS
    "Narrative":[
        "shopee games",
        "shopee game",
        "shopee tanam",
        "tanam",
        "shopee capit",
        "capit",
        "candy",
        "blockzi",
        "cocoki",
        "tebak kata",
        "nego neko",
        "fruity",
        "ceki ceki"
    ]
}

## Mapping Function

In [ ]:
# Mapping Function
def map_candidate_aspect(strong_anchor, weak_anchor):

    found = set()

    anchors = []

    if isinstance(strong_anchor, list):
        anchors.extend(strong_anchor)

    if isinstance(weak_anchor, list):
        anchors.extend(weak_anchor)

    anchors = [a.lower() for a in anchors]

    for aspect, keywords in aspect_mapping.items():

        for keyword in keywords:

            if keyword.lower() in anchors:

                found.add(aspect)

    if len(found) == 0:
        return ""

    return "; ".join(sorted(found))


candidate_df["suggested_aspect"] = candidate_df.apply(
    lambda row:
        map_candidate_aspect(
            row["strong_anchor"],
            row["weak_anchor"]
        ),
    axis=1
)

## Create Annotation Dataset

In [ ]:
annotation_df = candidate_df.copy()

annotation_df = annotation_df.reset_index(drop=True)

annotation_df.insert(
    0,
    "id",
    range(1, len(annotation_df)+1)
)

# Columns for annotator

annotation_df["final_gamification"] = ""

annotation_df["final_aspect"] = ""

annotation_df["sentiment"] = ""

annotation_df["notes"] = ""

# Column Structure

annotation_df = annotation_df[
    [
        "id",
        "review",
        "rating",
        "date",
        "strong_anchor",
        "weak_anchor",
        "suggested_aspect",
        "final_gamification",
        "final_aspect",
        "sentiment",
        "notes"
    ]
]

annotation_df.head()

,id,review,rating,date,strong_anchor,weak_anchor,suggested_aspect,final_gamification,final_aspect,sentiment,notes
0,1,saya mendapatkan 380 koin ya bos,5,2025-12-30 23:18:54,[koin],[],Points,,,,
1,2,Alhamdulillah dapat koin 50 semoga lebih besar...,5,2025-12-30 18:00:08,[koin],[],Points,,,,
2,3,cara mudah belanja dan mudah untuk mendapatkan...,5,2025-12-30 16:56:57,[koin],[],Points,,,,
3,4,"shopee skr payah ga kayak dulu, sekarang vouch...",1,2025-12-30 15:03:49,[vip],[voucher],Levels; Rewards,,,,
4,5,"saya masih berlangganan shoopie VIP,tp entah k...",4,2025-12-30 13:52:04,[vip],[],Levels,,,,


In [ ]:
# EXPORT TO EXCEL
SAVE_PATH = "/content/drive/MyDrive/ABSA_Shopee/05_manual_annotation.xlsx"

annotation_df.to_excel(
    SAVE_PATH,
    index=False
)

print("="*60)
print("Manual Annotation File Created Successfully")
print(SAVE_PATH)
print("="*60)

Manual Annotation File Created Successfully
/content/drive/MyDrive/ABSA_Shopee/05_manual_annotation.xlsx
